**Make sure you load the API keys for cloud providers!**

You can set your environment keys yourself or use a script. Please note that since keys are private, they are not included in the repository.

In [1]:
# setting the environment variables, the keys
import sys
import os

sys.path.insert(0, os.path.abspath('..'))

from config import set_environment
# for the keys - as explained early in chapter 2
set_environment()

# Basic RAG Implementation

In [3]:
# For Query Transformation
from langchain.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

# For Basic RAG Implementation
from langchain_community.document_loaders import JSONLoader
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# 1. Load Documents
loader = JSONLoader(
    file_path="knowledge_base.json",
    jq_schema=".[].content",
    text_content=True,
)

documents = loader.load()

# 2. Create Vectors
embedder = OpenAIEmbeddings(model="text-embedding-3-large")
embeddings = embedder.embed_documents([doc.page_content for doc in documents])

# 3 Store in Vector Store
vector_db = FAISS.from_documents(documents, embedder)

# 4. Retrieve similar docs
query = "What are the effects of climate change?"
results = vector_db.similarity_search(query)

In [4]:
print(len(results))
print(results)

4
[Document(id='16ce76c7-a70d-4dd4-bace-ca838fdb15e5', metadata={'source': '/Users/lesliepan/Documents/CAMICE/generative_ai_with_langchain/chapter4/knowledge_base.json', 'seq_num': 7}, page_content='Eating plant based alternatives have shown to reduce carbon footprint by 70% compared to eating beef.'), Document(id='69dfeaf0-5b46-4017-9e14-89be092ad922', metadata={'source': '/Users/lesliepan/Documents/CAMICE/generative_ai_with_langchain/chapter4/knowledge_base.json', 'seq_num': 6}, page_content='Prompt engineering involves designing and refining prompts to elicit desired responses from language models. Effective prompts can significantly improve the quality of generated text. Techniques include zero-shot prompting, few-shot prompting, and chain-of-thought prompting, where the model is guided through a series of reasoning steps.'), Document(id='6679baa3-44cc-4e75-9f6c-cc92b8f490d5', metadata={'source': '/Users/lesliepan/Documents/CAMICE/generative_ai_with_langchain/chapter4/knowledge_bas

# KNN Retriever

In [8]:
from langchain_community.retrievers import KNNRetriever
from langchain_openai import OpenAIEmbeddings

retriver = KNNRetriever.from_documents(documents, OpenAIEmbeddings())
results = retriver.invoke(query)

In [9]:
results

[Document(metadata={'source': '/Users/lesliepan/Documents/CAMICE/generative_ai_with_langchain/chapter4/knowledge_base.json', 'seq_num': 7}, page_content='Eating plant based alternatives have shown to reduce carbon footprint by 70% compared to eating beef.'),
 Document(metadata={'source': '/Users/lesliepan/Documents/CAMICE/generative_ai_with_langchain/chapter4/knowledge_base.json', 'seq_num': 1}, page_content="Transformer models were introduced in the paper 'Attention Is All You Need' by Vaswani et al. in 2017. The architecture relies on self-attention mechanisms rather than recurrent or convolutional neural networks. This design allows for more parallelization during training and better handling of long-range dependencies in text."),
 Document(metadata={'source': '/Users/lesliepan/Documents/CAMICE/generative_ai_with_langchain/chapter4/knowledge_base.json', 'seq_num': 6}, page_content='Prompt engineering involves designing and refining prompts to elicit desired responses from language m

# External Search API Retriever

In [10]:
from langchain_community.retrievers.pubmed import PubMedRetriever

retriever = PubMedRetriever()
results = retriever.invoke("COVID Research")

In [11]:
results

[Document(metadata={'uid': '41422309', 'Title': 'Insights into the self-assembly and interaction of sars-cov-2 fusion peptides with biomimetic plasma membranes.', 'Published': '2025-12-20', 'Copyright Information': '© 2025. The Author(s).'}, page_content='First identified in late 2019, the COVID-19 pandemic, caused by the SARS-CoV-2 coronavirus, rapidly escalated into a global health crisis. SARS-CoV-2 is a single-stranded RNA virus encased in a lipid envelope that houses key structural proteins, including the Spike glycoprotein, which mediates viral entry into host cells. Within Spike, the S2 subunit, and particularly its fusion domain, plays a critical role in merging viral and host membranes. To explore how receptor-driven Spike clustering influences this process, we investigated the self-assembly of S2 fusion peptides (FPs) and their interactions with biomimetic plasma membrane (PM) models composed of phospholipids, sphingomyelin, and cholesterol. Atomic force microscopy, laser dir